# EDA del balance de entradas y salidas de MiBici — 2023

Este notebook documenta la versión con fechas explícitas: ISO (`año-mes-día`) y fechas con barras (`día/mes/año`, utilizadas en noviembre). Analiza los viajes conservados por el flujo oficial de limpieza. Cada fila del Parquet horario representa una estación y una hora; el balance es **entradas − salidas**.

La lógica está en `mibici/plots.py`, reutilizable desde terminal o notebook. Se generan estadísticas, seis gráficas e interpretaciones y se registran en MLflow. El alcance es descriptivo: no se entrena un modelo ni se infiere disponibilidad real de bicicletas.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mibici.plots import run_eda

resultado = run_eda(
    input_path=ROOT / 'data/interim/hourly_balance_2023.parquet',
    features_path=ROOT / 'data/processed/features_2023.parquet',
    figures_dir=ROOT / 'reports/figures',
    report_dir=ROOT / 'reports/eda',
    station_id=51,
)
tablas = json.loads(Path(resultado['tablas']).read_text())
print('MLflow run_id:', resultado['run_id'])
print('Los hashes de los dos Parquet se registraron con esta ejecución.')

## 1. Volumen, distribución y significado de los ceros

Se comprueban totales y dimensiones. Las entradas y salidas globales coinciden porque se conservan solo viajes con inicio y fin dentro del año. **Balance cero** puede indicar igual número de llegadas y salidas con movimientos; **entradas y salidas simultáneamente cero** significa que no hay movimientos registrados. Los ceros no identifican cierres ni disponibilidad de una estación.

In [ ]:
display(pd.DataFrame.from_dict(resultado['metricas'], orient='index', columns=['valor']))
display(pd.DataFrame(tablas['estadisticas']))

## 2. Patrones horarios y días de la semana

Se comparan entradas y salidas de lunes a viernes frente al fin de semana. Las horas pico se observan en las curvas, sin fijarlas como resultado previo. Para comparar días de la semana se usa el **promedio de viajes por día**, evitando el efecto de que un día aparezca 52 o 53 veces en el año.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/01_flujo_horario_pico.png')))
display(Image(filename=str(ROOT / 'reports/figures/02_demanda_dia_semana.png')))
display(pd.DataFrame(tablas['dias_semana']))

## 3. Cambios entre meses

Se compara el promedio diario de viajes iniciados por mes para considerar su distinta duración. Los CSV se validan contra el mes de sus inicios de viaje. Las diferencias observadas no permiten atribuir causas como clima, cambios de oferta o cierres sin datos adicionales.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/06_flujo_mensual.png')))
display(pd.DataFrame(tablas['meses']))

## 4. Desbalance por estación

El ranking suma **|entradas − salidas| de cada hora**. Es distinto del balance neto acumulado: los desbalances positivos y negativos no se cancelan. El indicador también depende del volumen de uso y sirve para priorizar análisis; no demuestra urgencia de rebalanceo ni falta de bicicletas.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/03_top_estaciones_desbalance.png')))
display(pd.DataFrame(tablas['estaciones']).head(15))

## 5. Caso ilustrativo: estación 51

Se muestra su perfil medio de lunes a viernes. Un balance positivo representa más llegadas que salidas en esa hora y uno negativo, más salidas que llegadas. La estación 51 se mantiene para comparar con el análisis inicial del equipo; la gráfica no describe inventario acumulado.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/04_dinamica_estacion_51.png')))

## 6. Variables históricas y correlaciones

Las correlaciones se calculan **solo con enero–septiembre**, el tramo de entrenamiento previsto, e incluyen los retardos de 1, 24 y 168 horas y las medias móviles de las 6 horas anteriores. Las medias se calculan por estación, excluyendo la hora actual. La correlación no prueba causalidad ni precisión predictiva; la relación entre balance, entradas y salidas también depende de la definición del balance.

Se conserva la codificación cíclica de hora y día de la semana. No se aplican escalado ni PCA: el modelo de árboles existente no requiere escalado y se prioriza interpretar las variables. El identificador numérico de estación y la bandera de hora pico son decisiones provisionales que deberán evaluarse en modelado.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/05_matriz_correlacion_lags.png')))
print('Filas empleadas en la correlación:', resultado['metricas']['filas_correlacion_entrenamiento'])

## 7. Hallazgos, decisiones y límites

Las conclusiones siguientes se generan con las mismas tablas que las figuras. Los viajes con duración no positiva se excluyen; los atípicos por IQR y los viajes cortos o largos se diagnostican y conservan, porque una duración inusual no demuestra un error. El EDA completo de 2023 describe el histórico; no selecciona automáticamente variables ni acredita un pronóstico multihorizonte.

In [ ]:
display(Markdown(Path(resultado['hallazgos']).read_text()))

## Trazabilidad y fuentes

- Fuente: CSV de datos abiertos de MiBici contenidos en `mibici_raw_v1.tar.xz`, registrado como `raw_v1` en `manifests/mibici_original.json`.
- Las versiones actuales de los Parquet se identifican por SHA-256 y sus manifiestos en `manifests/`; el reporte de esta ejecución conserva los hashes de entrada.
- Las gráficas y tablas quedan en `reports/figures/` y `reports/eda/`, y como artefactos de MLflow.
- El conteo anterior de 3,908,646 viajes corresponde a una lectura que descartaba noviembre al no reconocer sus fechas. Esta versión exige lectura explícita y verifica el mes de cada archivo.
- Flujo inicial y EDA del equipo desarrollados por Leo; integración, revisión y ejecución local por Iván, con apoyo de OpenAI Codex para generación de código, depuración y documentación. La revisión y responsabilidad de la entrega corresponden al equipo.